In [1]:
import pandas as pd

# Create a synthetic dataset with some example sentences and corresponding sentiments
data = {
    "text": [
        "I love this product, it's amazing!",
        "This is the worst experience I've ever had.",
        "The movie was fantastic and exciting.",
        "I would not recommend this item to anyone.",
        "Great service, I will definitely come back!",
        "Absolutely horrible, do not buy this!",
        "The food was delicious and fresh.",
        "Worst purchase ever, complete waste of money.",
        "What a wonderful experience, very satisfied!",
        "The quality is terrible, I'm very disappointed."
    ],
    "sentiment": [1, 0, 1, 0, 1, 0, 1, 0, 1, 0]  # 1 for positive, 0 for negative
}

# Convert the dictionary to a DataFrame
df = pd.DataFrame(data)

# Display the dataset
print(df)


                                              text  sentiment
0               I love this product, it's amazing!          1
1      This is the worst experience I've ever had.          0
2            The movie was fantastic and exciting.          1
3       I would not recommend this item to anyone.          0
4      Great service, I will definitely come back!          1
5            Absolutely horrible, do not buy this!          0
6                The food was delicious and fresh.          1
7    Worst purchase ever, complete waste of money.          0
8     What a wonderful experience, very satisfied!          1
9  The quality is terrible, I'm very disappointed.          0


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Vectorizing the text using TF-IDF
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['text']).toarray()  # Convert to array for LSTM
y = df['sentiment']

# Split the dataset into training and testing sets (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training Data Shape: {X_train.shape}")
print(f"Test Data Shape: {X_test.shape}")


Training Data Shape: (8, 48)
Test Data Shape: (2, 48)


In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, Dropout
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Pad the sequences to make the input length uniform
X_train_pad = pad_sequences(X_train, padding='post', maxlen=500)
X_test_pad = pad_sequences(X_test, padding='post', maxlen=500)

# Build the LSTM model
model = Sequential()
model.add(Embedding(input_dim=5000, output_dim=128, input_length=500))  # Embedding layer for word embeddings
model.add(LSTM(128, return_sequences=True))  # LSTM layer
model.add(Dropout(0.5))  # Dropout layer to reduce overfitting
model.add(LSTM(64))  # Another LSTM layer
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))  # Output layer with sigmoid activation for binary classification

# Compile the model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model
history = model.fit(X_train_pad, y_train, epochs=10, batch_size=64, validation_data=(X_test_pad, y_test), verbose=1)


Epoch 1/10


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step - accuracy: 0.3750 - loss: 0.6919 - val_accuracy: 0.5000 - val_loss: 0.6932
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 719ms/step - accuracy: 0.2500 - loss: 0.6936 - val_accuracy: 0.5000 - val_loss: 0.6933
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.5000 - loss: 0.6980 - val_accuracy: 0.5000 - val_loss: 0.6935
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.5000 - loss: 0.6912 - val_accuracy: 0.5000 - val_loss: 0.6935
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.6250 - loss: 0.6857 - val_accuracy: 0.5000 - val_loss: 0.6934
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.5000 - loss: 0.7008 - val_accuracy: 0.5000 - val_loss: 0.6932
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.8750 - loss: 0.6695 - val_accuracy: 0.5000 - val_loss: 0.6932
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 1.0000 - loss: 0.6622 - val_accuracy: 0.5000 - val_loss: 0.6932
Epoch 9/10
1/1 ━━━━━━━━━

In [10]:
from sklearn.metrics import classification_report

# Evaluate the model on the test set
y_pred = model.predict(X_test_pad)
y_pred = (y_pred > 0.5).astype(int)  # Convert probabilities to binary values

# Print classification report
print(classification_report(y_test, y_pred))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 367ms/step
              precision    recall  f1-score   support

           0       0.50      1.00      0.67         1
           1       0.00      0.00      0.00         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [11]:
# Test with a new unseen review text
new_text = ["The product is absolutely terrible, not worth the money."]
new_text_tfidf = vectorizer.transform(new_text).toarray()
new_text_pad = pad_sequences(new_text_tfidf, padding='post', maxlen=500)

# Predict sentiment
pred = model.predict(new_text_pad)
sentiment = 'Positive' if pred > 0.5 else 'Negative'
print(f"Sentiment of the review: {sentiment}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 386ms/step
Sentiment of the review: Negative
